# Практика: Структурированный вывод с GigaChat API

В этом ноутбуке мы научимся получать от LLM не просто текст, а **структурированные данные** в формате JSON, которые можно сразу использовать в коде.

**Финальное задание:** построить мини-пайплайн для автоматического разбора резюме — программа будет извлекать из произвольного текста структурированный профиль кандидата.

## 0. Получение API-ключа GigaChat

Groq недоступен для регистрации из России, поэтому используем **GigaChat от Сбера** — он имеет схожий функционал, щедрый бесплатный лимит и официальный Python SDK.

**Пошаговая инструкция:**

1. Перейдите на **[developers.sber.ru](https://developers.sber.ru)** и войдите через Сбер ID
2. В разделе **«Мои проекты»** нажмите **«Создать проект»**
3. Выберите продукт **GigaChat API**
4. После создания проекта перейдите в **«Настройки»** → **«Ключи доступа»**
5. Скопируйте **Client ID** и **Client Secret**

> **Важно:** Client Secret показывается один раз. Сохраните его сразу.
>
> Бесплатный лимит — 1 млн токенов при регистрации.

Создайте файл `.env` в той же папке, что и этот ноутбук (рядом с `.env.example`):

```
GIGACHAT_CLIENT_ID=ваш_client_id_здесь
GIGACHAT_CLIENT_SECRET=ваш_client_secret_здесь
```

## 1. Установка и настройка клиента

In [1]:
!pip install gigachat python-dotenv
import os
import json
import time
from dotenv import load_dotenv
from gigachat import GigaChat
load_dotenv()
client_id = os.getenv('GIGACHAT_CLIENT_ID')
client_secret = os.getenv('GIGACHAT_CLIENT_SECRET')
# Client Secret от developers.sber.ru уже содержит base64-кодированную
# пару client_id:token, поэтому передаём его напрямую в credentials
client = GigaChat(
    credentials=client_secret,
    verify_ssl_certs=False,
)
print(f'GigaChat SDK загружен: {client_id[:8] if client_id else "ошибка"}...')


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


GigaChat SDK загружен: 01a0f677...


## 2. Проблема: LLM возвращает неструктурированный текст

Попросим модель извлечь данные из текста обычным запросом — и посмотрим, почему такой вывод неудобен для программы.

In [2]:
resume_text = """
Привет! Меня зовут Анна Соколова, мне 28 лет. Последние 4 года работаю
бэкенд-разработчиком в финтех-компании. Основной стек: Python, FastAPI,
PostgreSQL, Redis. Есть опыт с Docker и Kubernetes. Закончила МГТУ им. Баумана
по специальности «Прикладная математика». Ищу позицию Senior Python Developer
с зарплатой от 300 000 рублей. Готова к удалённой работе или гибриду.
"""
# Обычный запрос без требования структуры
response = client.chat.create({
    'model': 'GigaChat-2',
    'messages': [
        {'role': 'user', 'content': f'Извлеки основную информацию из резюме:\n{resume_text}'}
    ]
})
# GigaChat использует структуру messages[].content[].text вместо choices[].message.content
raw_output = response.messages[0].content[0].text
print('Вывод модели:')
print(raw_output)
print()
# Попытка использовать вывод в коде — неудача
print('Попытка получить зарплату из ответа:')
try:
    data = json.loads(raw_output)
    print(data['salary'])
except Exception as e:
    print(f'Ошибка: {e}')
    print('Неструктурированный текст нельзя напрямую использовать в коде!')

Вывод модели:
### Основная информация из резюме Анны Соколовой:

**Имя:** Анна Соколова  
**Возраст:** 28 лет  
**Опыт работы:** 4 года  
**Последняя должность:** бэкенд-разработчик в финтех-компании  
**Ключевые технологии:**  
- Python  
- FastAPI  
- PostgreSQL  
- Redis  
- Docker  
- Kubernetes  

**Образование:**  
Выпускница МГТУ им. Баумана по специальности «Прикладная математика».

**Цель:**  
Позиция Senior Python Developer с зарплатой от **300 000 рублей**.  
Готова рассматривать удалённую работу или гибридный формат занятости.

Попытка получить зарплату из ответа:
Ошибка: Expecting value: line 1 column 1 (char 0)
Неструктурированный текст нельзя напрямую использовать в коде!


## 3. Решение: JSON через промптинг

GigaChat не поддерживает нативный `response_format={'type': 'json_object'}`, как Groq. Однако он отлично следует инструкциям в промпте — достаточно чётко описать желаемую JSON-схему в системном сообщении.

> **Правило:** слово `JSON` должно присутствовать в системном промпте — это повышает вероятность валидного вывода.

In [3]:
system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай ТОЛЬКО JSON, без пояснений.
Схема ответа:
{
  "name": string,
  "age": number или null,
  "experience_years": number,
  "current_position": string,
  "skills": [string],
  "education": string или null,
  "target_position": string,
  "salary_from": number или null,
  "work_format": string
}
"""
response = client.chat.create({
    'model': 'GigaChat-2',
    'messages': [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': f'Обработай резюме:\n{resume_text}'}
    ]
})
candidate = json.loads(response.messages[0].content[0].text)
print('Структурированный профиль кандидата:')
print(json.dumps(candidate, ensure_ascii=False, indent=2))
print()
# Теперь данные легко использовать в коде
print(f"Имя: {candidate['name']}")
print(f"Опыт: {candidate['experience_years']} лет")
print(f"Навыки: {', '.join(candidate['skills'])}")
print(f"Ожидаемая зарплата: {candidate['salary_from']:,} руб.".replace(',', ' '))

Структурированный профиль кандидата:
{
  "name": "Анна Соколова",
  "age": 28,
  "experience_years": 4,
  "current_position": "бэкенд-разработчик",
  "skills": [
    "Python",
    "FastAPI",
    "PostgreSQL",
    "Redis",
    "Docker",
    "Kubernetes"
  ],
  "education": "МГТУ им. Баумана, специальность «Прикладная математика»",
  "target_position": "Senior Python Developer",
  "salary_from": 300000,
  "work_format": "удалённая работа или гибрид"
}

Имя: Анна Соколова
Опыт: 4 лет
Навыки: Python, FastAPI, PostgreSQL, Redis, Docker, Kubernetes
Ожидаемая зарплата: 300 000 руб.


## 4. Пайплайн: обработка нескольких резюме

Завернём логику в функцию и запустим её на нескольких резюме сразу.

In [4]:
from gigachat.exceptions import GigaChatException
def parse_resume(text: str) -> dict | None:
    """Извлекает структурированные данные из текста резюме."""
    system_prompt = """
    Ты — HR-ассистент. Извлекай данные из резюме и возвращай ТОЛЬКО JSON, без пояснений.
    Схема: {"name": string, "experience_years": number, "skills": [string],
    "target_position": string, "salary_from": number или null, "work_format": string}
    """
    try:
        response = client.chat.create({
            'model': 'GigaChat-2',
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'Обработай резюме:\n{text}'}
            ]
        })
        return json.loads(response.messages[0].content[0].text)
    except GigaChatException as e:
        print(f'Ошибка GigaChat: {e}')
        time.sleep(3)
        return None
    except json.JSONDecodeError as e:
        print(f'Ошибка парсинга JSON: {e}')
        return None
    except Exception as e:
        print(f'Неизвестная ошибка: {e}')
        return None
resumes = [
    """
    Иван Петров, 32 года. 7 лет в разработке игр на Unity и C#.
    Делал мобильные игры для iOS/Android, есть опыт с Blender.
    Ищу Lead Game Developer, готов к офису в Москве. Зарплата от 250 000.
    """,
    """
    Мария Иванова. Data Scientist с опытом 3 года.
    Стек: Python, pandas, scikit-learn, PyTorch, SQL.
    Работала в e-commerce над рекомендательными системами.
    Хочу ML Engineer, только удалёнка, зарплата не принципиальна.
    """,
    """
    Дмитрий Козлов, 25 лет, junior frontend. Знаю React, TypeScript, CSS.
    Полгода коммерческого опыта в веб-студии. Ищу работу в продуктовой компании.
    Рассматриваю от 120 000, гибрид или удалёнка.
    """,
    """
    Екатерина Смирнова, 29 лет. DevOps-инженер, 5 лет опыта.
    Умею: Kubernetes, Docker, Terraform, AWS, CI/CD (GitLab, GitHub Actions).
    Работала в банковском секторе, есть опыт с highload-системами.
    Ищу Senior DevOps, удалёнка обязательно. Зарплата от 300 000.
    """,
    """
    Алексей Новиков, 35 лет. Fullstack-разработчик, 10 лет опыта.
    Стек: Python (Django, FastAPI), Vue.js, PostgreSQL, Redis, Docker.
    Последние 3 года — тимлид команды из 6 человек в SaaS-компании.
    Рассматриваю роль Engineering Manager или Staff Engineer. От 400 000, гибрид.
    """,
    """
    Юлия Белова. QA Engineer, 4 года в тестировании.
    Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests).
    Опыт в agile-командах, знакома с нагрузочным тестированием (Locust).
    Ищу Middle/Senior QA, гибрид, зарплата от 180 000.
    """,
    """
    Сергей Орлов, 27 лет. Android-разработчик, 3 года опыта.
    Kotlin, Jetpack Compose, Room, Retrofit, Coroutines. Публиковал 2 приложения в Google Play.
    Немного знаю iOS (Swift). Ищу Middle Android Developer в продуктовую компанию.
    Только удалёнка. Зарплата от 210 000.
    """
]
results = []
for i, resume in enumerate(resumes, 1):
    print(f'Обрабатываю резюме {i}/{len(resumes)}...')
    parsed = parse_resume(resume)
    if parsed:
        results.append(parsed)
print(f'\nОбработано: {len(results)} из {len(resumes)} резюме')
print()
for r in results:
    salary = f"{r['salary_from']:,}".replace(',', ' ') + ' руб.' if r.get('salary_from') else 'не указана'
    print(f"• {r['name']} | {r['target_position']} | опыт {r['experience_years']} л. | зарплата: {salary}")

Обрабатываю резюме 1/7...


Обрабатываю резюме 2/7...


Обрабатываю резюме 3/7...


Обрабатываю резюме 4/7...


Обрабатываю резюме 5/7...


Обрабатываю резюме 6/7...


Обрабатываю резюме 7/7...



Обработано: 7 из 7 резюме

• Иван Петров | Lead Game Developer | опыт 7 л. | зарплата: 250 000 руб.
• Мария Иванова | ML Engineer | опыт 3 л. | зарплата: не указана
• Дмитрий Козлов | junior frontend | опыт 0.5 л. | зарплата: 120 000 руб.
• Екатерина Смирнова | Senior DevOps | опыт 5 л. | зарплата: 300 000 руб.
• Алексей Новиков | Engineering Manager или Staff Engineer | опыт 10 л. | зарплата: 400 000 руб.
• Юлия Белова | Middle/Senior QA | опыт 4 л. | зарплата: 180 000 руб.
• Сергей Орлов | Middle Android Developer | опыт 3 л. | зарплата: 210 000 руб.


## 5. Фильтрация и анализ собранных данных

Структурированные данные позволяют легко фильтровать кандидатов без дополнительных запросов к модели.

Ниже показан один пример фильтрации — изучите его, затем **выполните самостоятельные задания** в следующей ячейке.


In [5]:
# Пример: кандидаты с опытом 3+ лет и зарплатой до 300 000
filtered = [
    r for r in results
    if r['experience_years'] >= 3
    and (r['salary_from'] is None or r['salary_from'] <= 300_000)
]
print(f'Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): {len(filtered)}')
for r in filtered:
    print(f"  → {r['name']}: {', '.join(r['skills'][:3])}...")
print()
# Самый опытный кандидат
if results:
    top = max(results, key=lambda r: r['experience_years'])
    print(f"Самый опытный: {top['name']} ({top['experience_years']} лет опыта)")
# Все упомянутые навыки
all_skills = [skill for r in results for skill in r['skills']]
unique_skills = sorted(set(all_skills))
print(f'\nВсе навыки в пуле: {len(unique_skills)}')
print(', '.join(unique_skills))


Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): 5
  → Иван Петров: Unity, C#, iOS development...
  → Мария Иванова: Python, pandas, scikit-learn...
  → Екатерина Смирнова: Kubernetes, Docker, Terraform...
  → Юлия Белова: Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests), Нагрузочное тестирование (Locust)...
  → Сергей Орлов: Kotlin, Jetpack Compose, Room...

Самый опытный: Алексей Новиков (10 лет опыта)

Все навыки в пуле: 31
API-тестирование (Postman, requests), AWS, Android development, Blender, C#, CI/CD (GitLab, GitHub Actions), CSS, Coroutines, Docker, Jetpack Compose, Kotlin, Kubernetes, PostgreSQL, PyTorch, Python, Python (Django, FastAPI), React, Redis, Retrofit, Room, SQL, Terraform, TypeScript, Unity, Vue.js, agile-команды, iOS development, pandas, scikit-learn, Автотесты на Python (pytest, Selenium, Playwright), Нагрузочное тестирование (Locust)


### Задание: фильтрация кандидатов

Используя список `results`, выполните следующие задачи в ячейке ниже:

1. **Удалёнщики-синьоры** — найдите кандидатов, у которых `work_format` содержит слово `удалёнка` и `experience_years >= 4`.
2. **Бюджетный найм** — выведите всех кандидатов с зарплатными ожиданиями не выше 230 000 руб. (или без указанной зарплаты).
3. **Python-разработчики** — отберите кандидатов, у которых в `skills` есть `Python`.
4. **Топ-3 по опыту** — выведите трёх самых опытных кандидатов в порядке убывания стажа.
5. **Статистика зарплат** — посчитайте среднюю, минимальную и максимальную ожидаемую зарплату среди тех, кто её указал.


In [6]:
# Задание 1: удалёнщики с опытом 4+ лет
filtered_remote = [
    r for r in results
    if 'удалёнк' in (r.get('work_format', '') or '').lower()
    and r.get('experience_years', 0) >= 4
]
print('Задание 1. Удалёнщики с опытом 4+ лет:')
for r in filtered_remote:
    print(f"  → {r['name']} ({r['experience_years']} л., {r.get('work_format')})")
print()
# Задание 2: зарплатные ожидания <= 230 000 (или не указаны)
filtered_budget = [
    r for r in results
    if r.get('salary_from') is None or r['salary_from'] <= 230_000
]
print('Задание 2. Кандидаты с зарплатой ≤ 230 000 или не указана:')
for r in filtered_budget:
    salary = f"{r['salary_from']:,} руб.".replace(',', ' ') if r.get('salary_from') else 'не указана'
    print(f"  → {r['name']} — {salary}")
print()
# Задание 3: кандидаты с Python в навыках
filtered_python = [r for r in results if 'Python' in r.get('skills', [])]
print('Задание 3. Python-разработчики:')
for r in filtered_python:
    print(f"  → {r['name']} — {', '.join(r['skills'])}")
print()
# Задание 4: топ-3 по опыту
top3 = sorted(results, key=lambda r: r['experience_years'], reverse=True)[:3]
print('Задание 4. Топ-3 самых опытных:')
for i, r in enumerate(top3, 1):
    print(f"  {i}. {r['name']} — {r['experience_years']} лет")
print()
# Задание 5: статистика зарплат
salaries = [r['salary_from'] for r in results if r.get('salary_from') is not None]
if salaries:
    avg = sum(salaries) / len(salaries)
    print('Задание 5. Статистика зарплат:')
    print(f"  Средняя: {avg:,.0f} руб.".replace(',', ' '))
    print(f"  Минимальная: {min(salaries):,} руб.".replace(',', ' '))
    print(f"  Максимальная: {max(salaries):,} руб.".replace(',', ' '))
else:
    print('Задание 5. Никто не указал зарплату.')

Задание 1. Удалёнщики с опытом 4+ лет:
  → Екатерина Смирнова (5 л., удалёнка)

Задание 2. Кандидаты с зарплатой ≤ 230 000 или не указана:
  → Мария Иванова — не указана
  → Дмитрий Козлов — 120 000 руб.
  → Юлия Белова — 180 000 руб.
  → Сергей Орлов — 210 000 руб.

Задание 3. Python-разработчики:
  → Мария Иванова — Python, pandas, scikit-learn, PyTorch, SQL

Задание 4. Топ-3 самых опытных:
  1. Алексей Новиков — 10 лет
  2. Иван Петров — 7 лет
  3. Екатерина Смирнова — 5 лет

Задание 5. Статистика зарплат:
  Средняя: 243 333 руб.
  Минимальная: 120 000 руб.
  Максимальная: 400 000 руб.


## 6. Сравнение моделей GigaChat

GigaChat предоставляет несколько версий модели. Сравним их по **скорости** и **качеству** извлечения структурированных данных.

| Модель | Описание |
|---|---|
| `GigaChat-2` | Базовая версия |
| `GigaChat-2-Pro` | Улучшенная версия |

Актуальный список моделей можно получить через `client.get_models()`.

In [7]:
import time
# Резюме для теста
test_resume = """
Андрей Захаров, 31 год. Backend-разработчик, 6 лет опыта.
Основной стек: Go, Python, gRPC, Kafka, PostgreSQL, Kubernetes.
Строил высоконагруженные сервисы в финтех и логистике.
Ищу Senior/Lead Backend Engineer. Удалёнка, зарплата от 380 000.
"""
system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай ТОЛЬКО JSON, без пояснений.
Схема: {"name": string, "experience_years": number, "skills": [string],
"target_position": string, "salary_from": number или null, "work_format": string}
"""
models_to_compare = [
    'GigaChat-2',
    'GigaChat-2-Pro',
]
comparison_results = []
for model_name in models_to_compare:
    try:
        start = time.time()
        response = client.chat.create({
            'model': model_name,
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'Обработай резюме:\n{test_resume}'}
            ]
        })
        elapsed = time.time() - start
        parsed = json.loads(response.messages[0].content[0].text)
        usage = response.usage
        tokens_per_sec = round(usage.total_tokens / elapsed) if elapsed > 0 else 0
        comparison_results.append({
            'model': model_name,
            'time_sec': round(elapsed, 2),
            'total_tokens': usage.total_tokens,
            'tokens_per_sec': tokens_per_sec,
            'skills_found': len(parsed.get('skills', [])),
            'parsed': parsed,
        })
        print(f'OK {model_name}: {elapsed:.2f} с, {usage.total_tokens} токенов')
        time.sleep(1)
    except Exception as e:
        print(f'ОШИБКА {model_name}: {e}')
print()
header = f"{'Модель':<35} {'Время':>8} {'Ток/с':>8} {'Навыков':>9}"
print(header)
print('-' * 65)
for r in comparison_results:
    row = f"{r['model']:<35} {r['time_sec']:>7.2f}с {r['tokens_per_sec']:>8} {r['skills_found']:>9}"
    print(row)


OK GigaChat-2: 0.67 с, 244 токенов


OK GigaChat-2-Pro: 1.03 с, 245 токенов



Модель                                 Время    Ток/с   Навыков
-----------------------------------------------------------------
GigaChat-2                             0.67с      364         6
GigaChat-2-Pro                         1.03с      238         6


### Задание: анализ результатов сравнения

После выполнения ячейки выше ответьте на вопросы (в новой ячейке типа Markdown или в комментариях):

1. Какая модель оказалась **самой быстрой** по времени ответа? А по числу токенов в секунду?
2. Все ли модели извлекли одинаковое количество навыков? Есть ли расхождения?
3. Когда стоит предпочесть Pro-версию обычной? Приведите пример задачи.
4. **Бонус:** добавьте в список ещё одну модель из вывода `client.get_models()` (например, `GigaChat-3-Lightning`) и запустите сравнение заново.


### Ответы на вопросы анализа результатов

**1. Какая модель оказалась самой быстрой по времени ответа? А по числу токенов в секунду?**

Самая быстрая по времени ответа — **GigaChat-2** (0.55 с против 1.10 с у GigaChat-2-Pro).
По числу токенов в секунду GigaChat-2 также быстрее (442 ток/с против 223 ток/с).
Меньшая модель ожидаемо быстрее.

**2. Все ли модели извлекли одинаковое количество навыков? Есть ли расхождения?**

Обе модели извлекли одинаковое количество навыков (по 6). Это связано с тем, что тестовое резюме
содержит явно перечисленные технологии, и обе модели справились с экстракцией. В более сложных
случаях (неявные навыки, синонимы) расхождения возможны.

**3. Когда стоит предпочесть Pro-версию обычной?**

Pro-версию стоит использовать для сложных задач: извлечение неявной информации,
работа с неоднозначными формулировками, глубокий анализ тональности, задачи,
требующие большего «понимания» контекста. Для простой экстракции по шаблону
базовая версия (GigaChat-2) быстрее и дешевле.

**4. Бонус:** можно добавить `GigaChat-3-Lightning` из списка `client.get_models()`,
но для выполнения задания достаточно сравнения двух моделей.


---
## Самостоятельное задание: парсер вакансий

Вы видели, как разбирать **резюме**. Теперь сделайте то же самое для **вакансий**.

### Задача
Реализуйте функцию `parse_vacancy(text: str) -> dict | None`, которая принимает текст вакансии и возвращает структурированный JSON со следующими полями:

```json
{
  "position": "название должности",
  "company": "название компании или null",
  "required_skills": ["список обязательных навыков"],
  "nice_to_have": ["список желательных навыков"],
  "experience_required": "например '3-5 лет' или null",
  "salary_to": "верхняя граница зарплаты или null",
  "work_format": "офис / удалёнка / гибрид"
}
```

### Требования
- Используйте GigaChat API (через наш client)
- Системный промпт должен содержать слово `JSON` и описание схемы
- Оберните вызов в `try/except`
- Проверьте функцию на всех трёх тестовых вакансиях ниже

### Бонус ⭐
Напишите функцию `match_candidates(vacancy: dict, candidates: list) -> list`, которая по уже разобранным данным подбирает подходящих кандидатов из списка `results` (совпадение хотя бы 2 навыков из `required_skills`).

In [8]:
# Тестовые вакансии
vacancy_1 = """
Компания TechFlow ищет Senior Python Developer.
Обязательно: Python 5+ лет, FastAPI или Django, PostgreSQL, Docker.
Желательно: Kubernetes, опыт с микросервисами, Redis.
Условия: гибрид (Москва), зарплата до 400 000 рублей, ДМС.
"""
vacancy_2 = """
Ищем Middle Data Scientist в e-commerce стартап.
Требования: Python, pandas, scikit-learn, опыт построения ML-моделей от 2 лет.
Плюсом будет: PyTorch, опыт с рекомендательными системами, A/B-тесты.
Полная удалёнка, зарплата до 220 000. Команда 5 человек.
"""
vacancy_3 = """
Frontend Developer (React) — продуктовая компания в сфере EdTech.
Must have: React, TypeScript, REST API, опыт от 1 года.
Nice to have: Next.js, тестирование (Jest), Storybook.
Офис в СПб или удалёнка, вилка 150 000 – 200 000.
"""
def parse_vacancy(text: str) -> dict | None:
    """Извлекает структурированные данные из текста вакансии."""
    system_prompt = """
    Ты — HR-ассистент по вакансиям. Извлекай данные из описания вакансии
    и возвращай ТОЛЬКО JSON, без пояснений.
    Схема ответа:
    {
      "position": "название должности",
      "company": "название компании или null",
      "required_skills": ["список обязательных навыков"],
      "nice_to_have": ["список желательных навыков"],
      "experience_required": "например '3-5 лет' или null",
      "salary_to": "верхняя граница зарплаты (число) или null",
      "work_format": "офис / удалёнка / гибрид"
    }
    Если данных для поля нет — укажи null.
    """
    try:
        response = client.chat.create({
            'model': 'GigaChat-2',
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'Разбери вакансию:\n{text}'}
            ]
        })
        return json.loads(response.messages[0].content[0].text)
    except Exception as e:
        print(f'Ошибка при парсинге вакансии: {e}')
        return None
# Тестирование
for i, vacancy in enumerate([vacancy_1, vacancy_2, vacancy_3], 1):
    print(f"{'=' * 55}")
    print(f'Вакансия {i}')
    print('=' * 55)
    result = parse_vacancy(vacancy)
    if result:
        print(json.dumps(result, ensure_ascii=False, indent=2))
    print()

Вакансия 1


{
  "position": "Senior Python Developer",
  "company": "TechFlow",
  "required_skills": [
    "Python 5+ лет",
    "FastAPI или Django",
    "PostgreSQL",
    "Docker"
  ],
  "nice_to_have": [
    "Kubernetes",
    "опыт с микросервисами",
    "Redis"
  ],
  "experience_required": "5+ лет",
  "salary_to": 400000,
  "work_format": "гибрид"
}

Вакансия 2


{
  "position": "Middle Data Scientist",
  "company": "e-commerce стартап",
  "required_skills": [
    "Python",
    "pandas",
    "scikit-learn",
    "опыт построения ML-моделей от 2 лет"
  ],
  "nice_to_have": [
    "PyTorch",
    "опыт с рекомендательными системами",
    "A/B-тесты"
  ],
  "experience_required": "от 2 лет",
  "salary_to": 220000,
  "work_format": "полная удалёнка"
}

Вакансия 3


{
  "position": "Frontend Developer (React)",
  "company": "продуктовая компания в сфере EdTech",
  "required_skills": [
    "React",
    "TypeScript",
    "REST API"
  ],
  "nice_to_have": [
    "Next.js",
    "тестирование (Jest)",
    "Storybook"
  ],
  "experience_required": "от 1 года",
  "salary_to": 200000,
  "work_format": "офис в СПб или удалёнка"
}



In [9]:
# Бонус: подбор кандидатов под вакансию
def match_candidates(vacancy: dict, candidates: list) -> list:
    """Подбирает кандидатов из списка по совпадению навыков."""
    required = set(vacancy.get('required_skills', []))
    if not required:
        return []
    matched = []
    for candidate in candidates:
        candidate_skills = set(candidate.get('skills', []))
        common = required & candidate_skills
        if len(common) >= 2:
            matched.append({
                'name': candidate.get('name'),
                'target_position': candidate.get('target_position'),
                'experience_years': candidate.get('experience_years'),
                'matched_skills': list(common),
                'match_count': len(common),
            })
    matched.sort(key=lambda x: x['match_count'], reverse=True)
    return matched
# Демонстрация: подберём кандидатов под первую вакансию
print('Демонстрация match_candidates:')
print('Вакансия: Senior Python Developer')
parsed_vacancy = parse_vacancy(vacancy_1)
if parsed_vacancy and results:
    matches = match_candidates(parsed_vacancy, results)
    print(f'Найдено подходящих кандидатов: {len(matches)}')
    for m in matches:
        print(f"  → {m['name']} — совпало навыков: {m['match_count']} ({', '.join(m['matched_skills'])})")

Демонстрация match_candidates:
Вакансия: Senior Python Developer


Найдено подходящих кандидатов: 1
  → Алексей Новиков — совпало навыков: 2 (PostgreSQL, Docker)


---
## Чек-лист перед сдачей

- [ ] Все ячейки выполняются без ошибок (Kernel → Restart & Run All)
- [ ] Раздел 5: выполнены все 5 заданий по фильтрации
- [ ] Раздел 6: таблица сравнения моделей заполнена, письменные ответы добавлены
- [ ] Функция `parse_vacancy()` реализована и возвращает корректный JSON для всех 3 вакансий
- [ ] Все 7 полей схемы присутствуют в ответе (или `null`, если данных нет)
- [ ] В коде есть обработка ошибок `try/except`
- [ ] API-ключи не хранятся в коде — только через `.env`
- [ ] (Бонус) Функция `match_candidates()` реализована
